# 35. Reproducible prediction workflows
A reusable workflow checks what it receives. Missing columns, extra columns, wrong units, and invalid types can silently undermine predictions. Reproducibility also needs data access instructions, dependency records, seeds, and clear execution order. Predicting from a fresh process is stronger evidence than rerunning one cell in a long-lived notebook.

Define required columns and permitted types or values. Decide whether extra columns are rejected or ignored. Distinguish a deliberately missing value from an unparseable value. Write a small prediction script that validates before calling the loaded pipeline. Check it from a fresh environment using documented commands and an example input.

In [ ]:
import pandas as pd
required = ["hours", "distance"] # Define expected input columns.
def validate(frame):
    if set(frame.columns) != set(required): # Reject missing or unexpected columns.
        raise ValueError("Expected exactly hours and distance") # Explain the contract.
    result = frame[required].apply(pd.to_numeric, errors="raise") # Convert valid numeric values in fixed order.
    if result.isna().any().any(): # This simple schema disallows missing values.
        raise ValueError("Missing numeric input") # Reject incomplete records clearly.
    return result # Return a validated ordered table.
print(validate(pd.DataFrame({"distance": [3], "hours": [2]})).to_dict("records")) # Demonstrate order normalization.
try:
    validate(pd.DataFrame({"hours": [2]})) # Deliberately omit a required feature.
except ValueError as error:
    print("Rejected:", str(error)) # Show a useful validation message.
assert list(validate(pd.DataFrame({"hours": [2], "distance": [3]})).columns) == required # Check order.

## Track an experiment and identify its data
An experiment record connects settings, split definitions, code version, input data identity, environment, metrics and artifacts. A hash identifies exact bytes; it does not preserve those bytes or authorize sharing them. Git is useful for code, while data stores or a tool such as DVC can preserve permitted dataset versions. A container captures part of the environment but does not freeze the dataset, host hardware, remote services or all randomness. Begin with a transparent local record, then move to MLflow when searching and comparing many runs becomes cumbersome.

In [ ]:
import hashlib
import json
from pathlib import Path
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X, y = load_wine(return_X_y=True)
Xt, Xv, yt, yv = train_test_split(X, y, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=1000)).fit(Xt, yt)
record = {"dataset": "sklearn.load_wine", "data_sha256": hashlib.sha256(X.tobytes() + y.tobytes()).hexdigest(), "split_seed": 42, "parameters": {"C": 1}, "validation_accuracy": model.score(Xv, yv), "code_revision": "record-your-actual-git-commit"} # Replace the explicit code marker in a real project.
Path("experiment.json").write_text(json.dumps(record, indent=2)) # Write a portable local record.
loaded = json.loads(Path("experiment.json").read_text())
print("Recorded dataset:", loaded["dataset"])
print("Hash length:", len(loaded["data_sha256"]))
print("Validation accuracy:", round(loaded["validation_accuracy"], 3))
assert loaded["parameters"]["C"] == 1

### Reading the result
The 64-character SHA-256 string fingerprints these array bytes. Record shape, dtype and source too when moving beyond this demonstration. The local JSON is an experiment record, not a fully managed tracker. A real record needs the actual code revision and package manifest instead of the illustrative marker.

### Transfer exercise
Add a run manifest to an earlier project. Then follow the optional tooling guide to log the same parameters and metric in MLflow and version a permitted data file with DVC.

## Your turn
Write predict.py for your saved pipeline, plus a valid example input and two invalid cases. Document one command that reproduces predictions.

Try the task before opening the solution on the course website.